# System Resource Monitor

This notebook samples Windows system resources on a fixed interval for a fixed total
duration and writes the results to two Excel workbooks: a dashboard that always holds
the latest snapshot and a log that grows for the full run. Every row on every sheet of
both files carries the sample timestamp.

## Config

This is the only cell you adjust. It holds the sample interval, the total run duration,
and the two output file paths. Every later section reads from these values, so changing
them here changes the whole run. The defaults are a 5 minute interval over 24 hours,
which is 288 samples.

In [55]:
from datetime import timedelta
from pathlib import Path

# ----- Sampling -----

# How long to wait between samples.
SAMPLE_INTERVAL = timedelta(minutes=5)

# How long the whole run lasts. After this elapses the loop stops.
TOTAL_DURATION = timedelta(hours=24)

# ----- Output file paths (do not change these names) -----

DASHBOARD_PATH = Path("outputs") / "dashboard" / "dashboard.xlsx"
LOG_PATH = Path("outputs") / "log" / "log.xlsx"
CHARTS_DIR = Path("outputs") / "charts"
BENCHMARK_PATH = Path("outputs") / "benchmark.json"

# Make sure the output folders exist.
DASHBOARD_PATH.parent.mkdir(parents=True, exist_ok=True)
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

# ----- Alarm thresholds (a sample is flagged when a value crosses these) -----

CPU_ALARM_PERCENT = 90          # overall CPU usage above this is abnormal
MEMORY_ALARM_PERCENT = 90       # memory usage above this is abnormal
DISK_ALARM_PERCENT = 90         # any drive above this is abnormal
GPU_ALARM_PERCENT = 95          # GPU usage above this is abnormal
TEMPERATURE_ALARM_C = 85        # CPU temperature above this is abnormal
BATTERY_LOW_PERCENT = 15        # battery below this, on battery power, is abnormal

# Windows services that should always be running. A stopped one is flagged.
REQUIRED_SERVICES = ["Dnscache", "Schedule", "EventLog", "Winmgmt"]

# ----- Statistical baseline settings -----

# Fraction of the run treated as normal conditions to learn the benchmark from.
BASELINE_FRACTION = 0.25
# How many standard deviations away from the normal mean counts as a deviation.
BASELINE_STD_MULTIPLIER = 3

# Derived: total number of samples this run will take.
SAMPLE_COUNT = int(TOTAL_DURATION / SAMPLE_INTERVAL)

print("Sample interval :", SAMPLE_INTERVAL)
print("Total duration  :", TOTAL_DURATION)
print("Planned samples :", SAMPLE_COUNT)
print("Dashboard file  :", DASHBOARD_PATH)
print("Log file        :", LOG_PATH)
print("Charts folder   :", CHARTS_DIR)
print("Benchmark file  :", BENCHMARK_PATH)
print("Alarm limits    : CPU", CPU_ALARM_PERCENT, "Memory", MEMORY_ALARM_PERCENT,
      "Disk", DISK_ALARM_PERCENT, "GPU", GPU_ALARM_PERCENT,
      "Temp", TEMPERATURE_ALARM_C, "Battery low", BATTERY_LOW_PERCENT)
print("Required services:", REQUIRED_SERVICES)


Sample interval : 0:05:00
Total duration  : 1 day, 0:00:00
Planned samples : 288
Dashboard file  : outputs\dashboard\dashboard.xlsx
Log file        : outputs\log\log.xlsx
Charts folder   : outputs\charts
Benchmark file  : outputs\benchmark.json
Alarm limits    : CPU 90 Memory 90 Disk 90 GPU 95 Temp 85 Battery low 15
Required services: ['Dnscache', 'Schedule', 'EventLog', 'Winmgmt']


## Dependencies setup

This section makes the notebook run on a fresh machine that has nothing installed beyond
Python itself. It checks for each package the notebook needs and installs or upgrades only
what is missing or too old, so running it twice does no harm. The packages are `psutil`
for reading system resources and `openpyxl` for writing the Excel files.

A minimum version is enforced for each package, not just its presence. This matters
because some bundled Python setups (Anaconda for example) ship an old `psutil` that fails
on Python 3.12 when reading disks. The cell upgrades those automatically.

Two notes:

- Python 3 must already be installed, because the notebook needs Python to run at all. If
  you are on a machine with no Python, install Python 3 from python.org first, then open
  this notebook.
- If this cell reports that it upgraded or installed anything, restart the kernel and run
  the notebook again from the top, so the new version is the one actually loaded.


In [56]:
import importlib
import importlib.metadata as metadata
import subprocess
import sys

# Import name mapped to its pip name and the minimum version the notebook needs.
# matplotlib is used only at the end of the run to draw the trend graphs.
REQUIRED_PACKAGES = {
    "psutil": {"pip": "psutil", "min": "5.9.6"},
    "openpyxl": {"pip": "openpyxl", "min": "3.0.0"},
    "matplotlib": {"pip": "matplotlib", "min": "3.0.0"},
}


def _version_tuple(text):
    """Turn a version string like '5.9.6' into a comparable tuple of integers."""
    parts = []
    for chunk in text.split("."):
        digits = ""
        for char in chunk:
            if char.isdigit():
                digits += char
            else:
                break
        parts.append(int(digits) if digits else 0)
    return tuple(parts)


def _pip_install(spec):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", spec])


def ensure_packages(packages):
    """Install or upgrade each package so it is present and meets the minimum version."""
    print("Python version:", sys.version.split()[0])
    print("-" * 50)

    changed = False
    for import_name, info in packages.items():
        pip_name = info["pip"]
        minimum = info["min"]
        try:
            current = metadata.version(pip_name)
        except metadata.PackageNotFoundError:
            current = None

        if current is None:
            print("NOT INSTALLED  :", pip_name, "(need >=", minimum + ")")
            _pip_install(pip_name + ">=" + minimum)
            changed = True
        elif _version_tuple(current) < _version_tuple(minimum):
            print("OUTDATED       :", pip_name, current, "(need >=", minimum + ")")
            _pip_install(pip_name + ">=" + minimum)
            changed = True
        else:
            print("OK             :", pip_name, current)

    print("-" * 50)
    if changed:
        print("Packages were installed or upgraded.")
        print("Restart the kernel and run the notebook again from the top.")
    else:
        print("All required packages are present and up to date.")


ensure_packages(REQUIRED_PACKAGES)


Python version: 3.12.7
--------------------------------------------------
OK             : psutil 7.2.2
OK             : openpyxl 3.1.5
OK             : matplotlib 3.9.2
--------------------------------------------------
All required packages are present and up to date.


## CPU usage

This section reads CPU load. It returns the overall busy percentage and the busy
percentage of each logical core. The reading uses a one second measurement window
(`interval=1`) so the value reflects real activity rather than the zero you get from an
instant call. The overall figure is the mean of the per-core figures, so the two always
agree. The result is a flat dictionary of named fields plus the sample timestamp, which
is the shape every system-metrics collector in this notebook returns.

In [57]:
import psutil
from datetime import datetime


def collect_cpu(timestamp):
    """Return overall and per-core CPU usage for one sample, in plain labels."""
    per_core = psutil.cpu_percent(interval=1, percpu=True)
    overall = round(sum(per_core) / len(per_core), 1) if per_core else 0.0

    row = {
        "Time": timestamp,
        "Overall CPU Usage (%)": overall,
        "Number of CPU Cores": len(per_core),
    }
    for index, value in enumerate(per_core):
        # Cores numbered from 1 so the labels read naturally.
        row["Core {} Usage (%)".format(index + 1)] = value
    return row


# Quick check.
collect_cpu(datetime.now())


{'Time': datetime.datetime(2026, 6, 11, 14, 32, 33, 870886),
 'Overall CPU Usage (%)': 56.7,
 'Number of CPU Cores': 8,
 'Core 1 Usage (%)': 54.9,
 'Core 2 Usage (%)': 52.3,
 'Core 3 Usage (%)': 80.0,
 'Core 4 Usage (%)': 53.1,
 'Core 5 Usage (%)': 57.8,
 'Core 6 Usage (%)': 51.6,
 'Core 7 Usage (%)': 55.4,
 'Core 8 Usage (%)': 48.4}

## CPU temperature

This section reads the CPU thermal sensor. Windows does not expose CPU temperature
through psutil, so the reading is pulled from the WMI thermal zone using a short
PowerShell call. WMI reports the value in tenths of a Kelvin, which this code converts to
degrees Celsius. Many laptops block this sensor or return access denied. When that
happens the field is recorded as `Unavailable` for that sample rather than letting the
error stop the run, so the timestamp and the rest of the sample are still captured.

In [58]:
import subprocess
from datetime import datetime


def _run_powershell(command, timeout=15):
    """Run a PowerShell command and return its text output, or None on failure."""
    try:
        result = subprocess.run(
            ["powershell", "-NoProfile", "-NonInteractive", "-Command", command],
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        if result.returncode == 0:
            return result.stdout.strip()
    except Exception:
        pass
    return None


def collect_temperature(timestamp):
    """Return the CPU temperature in Celsius, or Unavailable if the sensor is blocked."""
    command = (
        "Get-CimInstance -Namespace root/wmi "
        "-ClassName MSAcpi_ThermalZoneTemperature -ErrorAction Stop "
        "| Select-Object -ExpandProperty CurrentTemperature"
    )
    output = _run_powershell(command)

    temperature = "Unavailable"
    if output:
        readings = []
        for line in output.splitlines():
            line = line.strip()
            if line.isdigit():
                # WMI reports tenths of a Kelvin. Convert to Celsius.
                readings.append(int(line) / 10.0 - 273.15)
        if readings:
            temperature = round(sum(readings) / len(readings), 1)

    return {
        "Time": timestamp,
        "CPU Temperature (°C)": temperature,
    }


# Quick check.
collect_temperature(datetime.now())

{'Time': datetime.datetime(2026, 6, 11, 14, 32, 34, 889372),
 'CPU Temperature (°C)': 'Unavailable'}

## RAM

This section reads system memory. It reports the total memory fitted, how much is in use,
how much is still available, and the used percentage. The raw figures from the system are
in bytes, which are hard to read, so they are converted to gigabytes (GB) and rounded.
Available memory is the amount that can be handed to programs without swapping, which is
the number most people care about when asking how much memory is free.

In [59]:
import psutil
from datetime import datetime

BYTES_PER_GB = 1024 ** 3


def to_gb(value):
    """Convert a byte count to gigabytes, rounded to two decimals."""
    return round(value / BYTES_PER_GB, 2)


def collect_ram(timestamp):
    """Return total, used, available memory in GB and the used percentage."""
    memory = psutil.virtual_memory()
    return {
        "Time": timestamp,
        "Total Memory (GB)": to_gb(memory.total),
        "Used Memory (GB)": to_gb(memory.used),
        "Available Memory (GB)": to_gb(memory.available),
        "Memory Usage (%)": memory.percent,
    }


# Quick check.
collect_ram(datetime.now())

{'Time': datetime.datetime(2026, 6, 11, 14, 32, 35, 335361),
 'Total Memory (GB)': 7.71,
 'Used Memory (GB)': 6.8,
 'Available Memory (GB)': 0.91,
 'Memory Usage (%)': 88.2}

## Disk

This section reads storage usage for every drive on the machine. Unlike the single
system-metrics row, disk returns one row per drive, so it gets its own sheet in both
Excel files. Each row carries the sample timestamp, the drive letter, total size, used
space, free space (all in gigabytes), and the used percentage. Some entries such as an
empty CD drive or card reader raise an error when read. Those are skipped so they do not
stop the sample, and the drives that do respond are still recorded.

In [60]:
import psutil
from datetime import datetime

BYTES_PER_GB = 1024 ** 3


def _gb(value):
    return round(value / BYTES_PER_GB, 2)


def collect_disk(timestamp):
    """Return one row per drive with total, used, free space and used percentage."""
    rows = []
    for partition in psutil.disk_partitions(all=False):
        try:
            usage = psutil.disk_usage(partition.mountpoint)
        except Exception:
            # Empty CD drive, card reader, or a drive psutil cannot read on Windows
            # (this can raise PermissionError, OSError, or SystemError). Skip it so
            # one bad drive does not stop the sample.
            continue
        rows.append({
            "Time": timestamp,
            "Drive": partition.device,
            "Total Space (GB)": _gb(usage.total),
            "Used Space (GB)": _gb(usage.used),
            "Free Space (GB)": _gb(usage.free),
            "Disk Usage (%)": usage.percent,
        })
    return rows


# Quick check.
collect_disk(datetime.now())


[{'Time': datetime.datetime(2026, 6, 11, 14, 32, 35, 356665),
  'Drive': 'C:\\',
  'Total Space (GB)': 476.23,
  'Used Space (GB)': 354.19,
  'Free Space (GB)': 122.05,
  'Disk Usage (%)': 74.4},
 {'Time': datetime.datetime(2026, 6, 11, 14, 32, 35, 356665),
  'Drive': 'D:\\',
  'Total Space (GB)': 237.86,
  'Used Space (GB)': 72.83,
  'Free Space (GB)': 165.03,
  'Disk Usage (%)': 30.6}]

## Battery

This section reads the battery. It reports the charge percentage, whether the laptop is
plugged in, and the estimated time left on battery. The estimate is given by the system in
seconds, which is converted here to a plain reading like `2 hours 15 minutes`. When the
machine is plugged in or the system cannot estimate the time, that is stated in words
rather than shown as a confusing number. On a desktop with no battery the fields read
`No battery` so the row still records cleanly with its timestamp.

In [61]:
import psutil
from datetime import datetime


def _format_time_left(seconds):
    """Turn a seconds-remaining value into a plain reading."""
    if seconds == psutil.POWER_TIME_UNLIMITED:
        return "Plugged in"
    if seconds == psutil.POWER_TIME_UNKNOWN or seconds is None or seconds < 0:
        return "Unknown"
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    return "{} hours {} minutes".format(hours, minutes)


def collect_battery(timestamp):
    """Return battery charge, plugged-in state and estimated time remaining."""
    battery = psutil.sensors_battery()
    if battery is None:
        return {
            "Time": timestamp,
            "Battery Charge (%)": "No battery",
            "Power Plugged In": "No battery",
            "Estimated Time Remaining": "No battery",
        }
    return {
        "Time": timestamp,
        "Battery Charge (%)": round(battery.percent, 1),
        "Power Plugged In": "Yes" if battery.power_plugged else "No",
        "Estimated Time Remaining": (
            "Plugged in" if battery.power_plugged
            else _format_time_left(battery.secsleft)
        ),
    }


# Quick check.
collect_battery(datetime.now())

{'Time': datetime.datetime(2026, 6, 11, 14, 32, 35, 373195),
 'Battery Charge (%)': 81,
 'Power Plugged In': 'No',
 'Estimated Time Remaining': '1 hours 20 minutes'}

## Network

This section reads network traffic for every network interface, so it returns one row per
interface and gets its own sheet. The system only reports running totals since the machine
booted, so this code remembers the previous sample's totals and reports the difference,
which is the data actually sent and received during the last interval. Both the per-sample
amounts and the running totals are shown, converted from bytes to megabytes (MB) for
readability. On the very first sample there is no previous total to compare against, so the
per-sample amounts read as 0.0 by design.

In [62]:
import psutil
from datetime import datetime

BYTES_PER_MB = 1024 ** 2

# Remembers the previous sample's byte counters per interface, so we can report
# the difference rather than the running total. Persists between samples.
_previous_net_counters = {}


def _mb(value):
    return round(value / BYTES_PER_MB, 3)


def collect_network(timestamp):
    """Return per-interface data sent and received since the last sample, plus totals."""
    rows = []
    counters = psutil.net_io_counters(pernic=True)
    for interface, stats in counters.items():
        previous = _previous_net_counters.get(interface)
        if previous is None:
            sent_delta = 0
            recv_delta = 0
        else:
            sent_delta = max(0, stats.bytes_sent - previous.bytes_sent)
            recv_delta = max(0, stats.bytes_recv - previous.bytes_recv)
        rows.append({
            "Time": timestamp,
            "Interface": interface,
            "Data Sent Since Last Sample (MB)": _mb(sent_delta),
            "Data Received Since Last Sample (MB)": _mb(recv_delta),
            "Total Data Sent (MB)": _mb(stats.bytes_sent),
            "Total Data Received (MB)": _mb(stats.bytes_recv),
        })
        _previous_net_counters[interface] = stats
    return rows


# Quick check. Run it twice to see real per-sample amounts on the second call.
collect_network(datetime.now())

[{'Time': datetime.datetime(2026, 6, 11, 14, 32, 35, 391128),
  'Interface': 'Ethernet',
  'Data Sent Since Last Sample (MB)': 0.0,
  'Data Received Since Last Sample (MB)': 0.0,
  'Total Data Sent (MB)': 0.0,
  'Total Data Received (MB)': 0.0},
 {'Time': datetime.datetime(2026, 6, 11, 14, 32, 35, 391128),
  'Interface': 'Ethernet 2',
  'Data Sent Since Last Sample (MB)': 0.0,
  'Data Received Since Last Sample (MB)': 0.0,
  'Total Data Sent (MB)': 0.0,
  'Total Data Received (MB)': 0.0},
 {'Time': datetime.datetime(2026, 6, 11, 14, 32, 35, 391128),
  'Interface': 'Local Area Connection* 1',
  'Data Sent Since Last Sample (MB)': 0.0,
  'Data Received Since Last Sample (MB)': 0.0,
  'Total Data Sent (MB)': 0.0,
  'Total Data Received (MB)': 0.0},
 {'Time': datetime.datetime(2026, 6, 11, 14, 32, 35, 391128),
  'Interface': 'Local Area Connection* 2',
  'Data Sent Since Last Sample (MB)': 0.0,
  'Data Received Since Last Sample (MB)': 0.0,
  'Total Data Sent (MB)': 0.0,
  'Total Data Rece

## GPU

This section reads graphics card usage. This machine has Intel Iris Xe integrated
graphics and no NVIDIA tooling, so the usual `nvidia-smi` route does not apply. Instead it
reads the Windows GPU performance counters through a short PowerShell call, which works for
any GPU vendor including Intel. GPU usage is the combined activity across the graphics
engines, capped at 100 percent, and GPU memory used is the memory currently held by
processes, shown in megabytes. If the counters cannot be read the fields record
`Unavailable` for that sample so the run continues. This reuses the PowerShell helper
defined in the CPU temperature section, so run the notebook from the top.


In [63]:
from datetime import datetime

# PowerShell that sums GPU engine utilization and GPU process memory across all
# instances. Each value is printed on its own labelled line, or NA if unreadable.
_GPU_COMMAND = r"""
try {
    $u = ((Get-Counter '\GPU Engine(*)\Utilization Percentage' -ErrorAction Stop).CounterSamples |
          Measure-Object -Property CookedValue -Sum).Sum
} catch { $u = 'NA' }
try {
    $m = ((Get-Counter '\GPU Process Memory(*)\Local Usage' -ErrorAction Stop).CounterSamples |
          Measure-Object -Property CookedValue -Sum).Sum
} catch { $m = 'NA' }
Write-Output ("UTIL=" + $u)
Write-Output ("MEM=" + $m)
"""


def collect_gpu(timestamp):
    """Return GPU usage percent and GPU memory used in MB, or Unavailable."""
    output = _run_powershell(_GPU_COMMAND, timeout=20)

    usage = "Unavailable"
    memory_mb = "Unavailable"
    if output:
        for line in output.splitlines():
            line = line.strip()
            if line.startswith("UTIL="):
                value = line[5:].strip().replace(",", ".")
                try:
                    usage = min(100.0, round(float(value), 1))
                except ValueError:
                    pass
            elif line.startswith("MEM="):
                value = line[4:].strip().replace(",", ".")
                try:
                    memory_mb = round(float(value) / (1024 ** 2), 1)
                except ValueError:
                    pass

    return {
        "Time": timestamp,
        "GPU Usage (%)": usage,
        "GPU Memory Used (MB)": memory_mb,
    }


# Quick check.
collect_gpu(datetime.now())


{'Time': datetime.datetime(2026, 6, 11, 14, 32, 35, 419164),
 'GPU Usage (%)': 4.6,
 'GPU Memory Used (MB)': 961.8}

## System uptime and boot time

This section reports when the machine last started up and how long it has been running
since. The system gives the boot time as a raw timestamp, which is converted here to a
plain date and time. Uptime is the gap between now and that boot time, shown in days,
hours and minutes so it is easy to read at a glance.

In [64]:
import psutil
from datetime import datetime


def _format_uptime(seconds):
    """Turn a number of seconds into days, hours and minutes."""
    seconds = int(seconds)
    days = seconds // 86400
    hours = (seconds % 86400) // 3600
    minutes = (seconds % 3600) // 60
    return "{} days {} hours {} minutes".format(days, hours, minutes)


def collect_uptime(timestamp):
    """Return the boot time and how long the machine has been running."""
    boot = datetime.fromtimestamp(psutil.boot_time())
    uptime_seconds = (timestamp - boot).total_seconds()
    return {
        "Time": timestamp,
        "Last Boot Time": boot.strftime("%Y-%m-%d %H:%M:%S"),
        "Uptime": _format_uptime(uptime_seconds),
    }


# Quick check.
collect_uptime(datetime.now())

{'Time': datetime.datetime(2026, 6, 11, 14, 32, 38, 367100),
 'Last Boot Time': '2026-06-10 08:22:40',
 'Uptime': '1 days 6 hours 9 minutes'}

## Processes

This section lists every program currently running, so it returns one row per process and
gets its own sheet. Each row shows the process id, the program name, how much CPU it is
using, how much memory it holds in megabytes, and its status such as running or sleeping.
Some processes are protected by Windows and cannot be read; those are skipped so they do
not stop the sample. The CPU figure is measured since the previous sample, so on the very
first sample it reads 0.0 for every process, then shows real values from the second sample
onward. To keep the check below short it prints only the first five rows and the total
count.

In [65]:
import psutil
from datetime import datetime

BYTES_PER_MB = 1024 ** 2

# Number of logical cores. Used to convert per-process CPU into a 0 to 100 share of
# total CPU capacity, the way Task Manager shows it, instead of a per-core figure that
# can climb above 100 on a multi-core machine.
CPU_CORE_COUNT = psutil.cpu_count() or 1


def collect_processes(timestamp):
    """Return one row per running process with cpu, memory and status."""
    rows = []
    for proc in psutil.process_iter(["pid", "name", "cpu_percent", "memory_info", "status"]):
        try:
            info = proc.info
            memory = info.get("memory_info")
            memory_mb = round(memory.rss / BYTES_PER_MB, 1) if memory else 0.0
            name = info.get("name") or "Unknown"
            cpu_share = (info.get("cpu_percent") or 0.0) / CPU_CORE_COUNT
            rows.append({
                "Time": timestamp,
                "Process ID (PID)": info.get("pid"),
                "Process Name": name,
                "CPU Usage (%)": round(cpu_share, 1),
                "Memory Usage (MB)": memory_mb,
                "Status": info.get("status") or "Unknown",
            })
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            # Process ended or is protected by Windows. Skip it.
            continue
    return rows


# Quick check: show the first five rows and the total count.
_rows = collect_processes(datetime.now())
print("Total processes:", len(_rows))
_rows[:5]


Total processes: 365


[{'Time': datetime.datetime(2026, 6, 11, 14, 32, 38, 386810),
  'Process ID (PID)': 0,
  'Process Name': 'System Idle Process',
  'CPU Usage (%)': 72.0,
  'Memory Usage (MB)': 0.0,
  'Status': 'running'},
 {'Time': datetime.datetime(2026, 6, 11, 14, 32, 38, 386810),
  'Process ID (PID)': 4,
  'Process Name': 'System',
  'CPU Usage (%)': 0.8,
  'Memory Usage (MB)': 0.1,
  'Status': 'running'},
 {'Time': datetime.datetime(2026, 6, 11, 14, 32, 38, 386810),
  'Process ID (PID)': 104,
  'Process Name': 'Unknown',
  'CPU Usage (%)': 0.0,
  'Memory Usage (MB)': 23.3,
  'Status': 'stopped'},
 {'Time': datetime.datetime(2026, 6, 11, 14, 32, 38, 386810),
  'Process ID (PID)': 164,
  'Process Name': 'Registry',
  'CPU Usage (%)': 0.0,
  'Memory Usage (MB)': 39.1,
  'Status': 'running'},
 {'Time': datetime.datetime(2026, 6, 11, 14, 32, 38, 386810),
  'Process ID (PID)': 324,
  'Process Name': 'bash.exe',
  'CPU Usage (%)': 0.0,
  'Memory Usage (MB)': 1.2,
  'Status': 'running'}]

## Windows services

This section lists every Windows service, so it returns one row per service and gets its
own sheet. Services are background programs the system runs, separate from the apps you
open. Each row shows the short service name, the friendly display name, whether it is
running or stopped, and its start type such as automatic or manual. A service that cannot
be read is skipped so it does not stop the sample. The check below prints only the first
five rows and the total count.

In [66]:
import psutil
from datetime import datetime


def collect_services(timestamp):
    """Return one row per Windows service with status and start type."""
    rows = []
    for service in psutil.win_service_iter():
        try:
            info = service.as_dict()
            rows.append({
                "Time": timestamp,
                "Service Name": info.get("name") or "Unknown",
                "Display Name": info.get("display_name") or "Unknown",
                "Status": info.get("status") or "Unknown",
                "Start Type": info.get("start_type") or "Unknown",
            })
        except Exception:
            # Service could not be read. Skip it so the sample continues.
            continue
    return rows


# Quick check: show the first five rows and the total count.
_rows = collect_services(datetime.now())
print("Total services:", len(_rows))
_rows[:5]

Total services: 324


[{'Time': datetime.datetime(2026, 6, 11, 14, 32, 41, 262653),
  'Service Name': 'AJRouter',
  'Display Name': 'AllJoyn Router Service',
  'Status': 'stopped',
  'Start Type': 'manual'},
 {'Time': datetime.datetime(2026, 6, 11, 14, 32, 41, 262653),
  'Service Name': 'ALG',
  'Display Name': 'Application Layer Gateway Service',
  'Status': 'stopped',
  'Start Type': 'manual'},
 {'Time': datetime.datetime(2026, 6, 11, 14, 32, 41, 262653),
  'Service Name': 'AppIDSvc',
  'Display Name': 'Application Identity',
  'Status': 'stopped',
  'Start Type': 'manual'},
 {'Time': datetime.datetime(2026, 6, 11, 14, 32, 41, 262653),
  'Service Name': 'Appinfo',
  'Display Name': 'Application Information',
  'Status': 'running',
  'Start Type': 'manual'},
 {'Time': datetime.datetime(2026, 6, 11, 14, 32, 41, 262653),
  'Service Name': 'AppMgmt',
  'Display Name': 'Application Management',
  'Status': 'stopped',
  'Start Type': 'manual'}]

## Excel writing logic

This section turns the collected data into the two Excel files. Both files use the same
five sheets: system_metrics, disk, network, processes and services. Every row on every
sheet carries the sample timestamp.

There are two writers:

- `write_dashboard` rebuilds the whole workbook from scratch every sample and overwrites
  `outputs/dashboard/dashboard.xlsx`. The dashboard always shows only the latest snapshot,
  so the system_metrics sheet holds a single row and the others hold the current lists.
- `append_log` adds this sample to `outputs/log/log.xlsx`. On the first sample it creates
  the file with a header row on each sheet. On every later sample it opens the existing
  file and appends the new rows below, so the log grows for the full run and is never
  overwritten. Appended rows are aligned to the header that was written first, so the
  columns stay consistent.

A helper `_build_datasets` runs every collector for one sample and groups the results by
sheet name. The system_metrics row is built by merging the CPU, temperature, RAM, battery,
GPU and uptime readings into one wide row that shares a single timestamp.

The check below writes one real sample to both files, reads them back to confirm the
sheets and row counts, then deletes the two test files so the outputs folders are clean
for the real run.


In [67]:
import os
from datetime import datetime

from openpyxl import Workbook, load_workbook

# The five sheets used in both the dashboard and the log.
SHEET_NAMES = ["system_metrics", "disk", "network", "processes", "services"]


def _build_datasets(timestamp):
    """Run every collector for one sample and group results by sheet name."""
    system_row = {}
    for collector in (
        collect_cpu,
        collect_temperature,
        collect_ram,
        collect_battery,
        collect_gpu,
        collect_uptime,
    ):
        system_row.update(collector(timestamp))

    return {
        "system_metrics": [system_row],
        "disk": collect_disk(timestamp),
        "network": collect_network(timestamp),
        "processes": collect_processes(timestamp),
        "services": collect_services(timestamp),
    }


def _sheet_is_empty(ws):
    """True when a worksheet has no header row yet.

    This checks the sheet dimensions instead of reading a cell such as ws['A1']. On a
    freshly created sheet, reading a cell instantiates it and pushes the first append
    down to row 2, leaving a blank first row that then breaks later appends. Reading
    max_row and max_column does not touch any cell, so it stays clean.
    """
    return ws.max_row == 1 and ws.max_column == 1


def write_dashboard(datasets):
    """Rebuild the dashboard workbook from scratch and overwrite it (latest snapshot)."""
    wb = Workbook()
    wb.remove(wb.active)
    for name in SHEET_NAMES:
        ws = wb.create_sheet(title=name)
        rows = datasets.get(name) or []
        if rows:
            headers = list(rows[0].keys())
            ws.append(headers)
            for row in rows:
                ws.append([row.get(header) for header in headers])
        else:
            ws.append(["Time", "No data this sample"])
    wb.save(DASHBOARD_PATH)


def append_log(datasets):
    """Append this sample to the log workbook, creating it on the first sample."""
    if LOG_PATH.exists():
        wb = load_workbook(LOG_PATH)
    else:
        wb = Workbook()
        wb.remove(wb.active)

    for name in SHEET_NAMES:
        ws = wb[name] if name in wb.sheetnames else wb.create_sheet(title=name)
        rows = datasets.get(name) or []
        if not rows:
            continue
        if _sheet_is_empty(ws):
            headers = list(rows[0].keys())
            ws.append(headers)
        else:
            headers = [cell.value for cell in ws[1]]
        for row in rows:
            ws.append([row.get(header) for header in headers])
    wb.save(LOG_PATH)


# Quick check: write one real sample to both files, read them back, then remove the
# test files so the outputs folders stay clean for the real run.
_datasets = _build_datasets(datetime.now())
write_dashboard(_datasets)
append_log(_datasets)

for _path in (DASHBOARD_PATH, LOG_PATH):
    _wb = load_workbook(_path)
    print(_path.name, "sheets:", _wb.sheetnames)
    for _name in SHEET_NAMES:
        print("   ", _name, "rows:", _wb[_name].max_row)
    _wb.close()

os.remove(DASHBOARD_PATH)
os.remove(LOG_PATH)
print("Test files removed. Outputs are clean for the real run.")


dashboard.xlsx sheets: ['system_metrics', 'disk', 'network', 'processes', 'services']
    system_metrics rows: 2
    disk rows: 3
    network rows: 8
    processes rows: 366
    services rows: 325
log.xlsx sheets: ['system_metrics', 'disk', 'network', 'processes', 'services', 'alarms']
    system_metrics rows: 5
    disk rows: 9
    network rows: 29
    processes rows: 1467
    services rows: 1297
Test files removed. Outputs are clean for the real run.


## Main collection loop

This section is the engine. The function `run_monitor` takes a sample, writes the
dashboard and appends to the log, waits for the interval, and repeats until the total
duration has passed. It reads the interval and duration from the config section, so the
real run is every 5 minutes for 24 hours.

Each run starts fresh: any dashboard or log file from a previous run is removed first, so
one run produces one clean log. The network counters are also reset so the first sample's
per-sample data reads as 0.0 and grows from there. Every sample is wrapped so that one bad
sample prints a short message and the run carries on rather than stopping.

This function only collects and records. The benchmark, the alarms and the charts run at
the end of the run, in the sections below, and are tied together in the final run cell.

The check below does a short burst of a few samples at a 10 second interval instead of the
real 5 minutes, so you can confirm the loop works without waiting. It leaves a small log
behind on purpose, so the benchmark and alarm sections further down have real data to work
with while we build them.


In [68]:
import time
from datetime import datetime, timedelta

from openpyxl import load_workbook


def run_monitor(interval=None, duration=None, fresh_start=True, verbose=True):
    """Sample resources on a fixed interval for a fixed duration, writing both files.

    interval and duration default to the config values (5 minutes, 24 hours). A short
    interval and duration can be passed for testing. fresh_start removes any existing
    dashboard and log first so each run produces one clean log.
    """
    interval = interval if interval is not None else SAMPLE_INTERVAL
    duration = duration if duration is not None else TOTAL_DURATION

    if fresh_start:
        for path in (DASHBOARD_PATH, LOG_PATH):
            if path.exists():
                path.unlink()

    # Reset the network baseline so the first sample's per-sample data starts at zero.
    _previous_net_counters.clear()

    start = datetime.now()
    end = start + duration
    next_time = start
    sample_number = 0

    print("Run started at", start.strftime("%Y-%m-%d %H:%M:%S"))
    print("Interval:", interval, " Duration:", duration)
    print("-" * 50)

    while datetime.now() < end:
        sample_number += 1
        timestamp = datetime.now()
        try:
            datasets = _build_datasets(timestamp)
            write_dashboard(datasets)
            append_log(datasets)
            if verbose:
                system_row = datasets["system_metrics"][0]
                print("Sample {} at {}   CPU {}%   Memory {}%".format(
                    sample_number,
                    timestamp.strftime("%H:%M:%S"),
                    system_row.get("Overall CPU Usage (%)"),
                    system_row.get("Memory Usage (%)"),
                ))
        except Exception as error:
            print("Sample", sample_number, "failed:", error)

        # Keep a steady cadence: sleep until the next scheduled sample time.
        next_time += interval
        sleep_for = (next_time - datetime.now()).total_seconds()
        if sleep_for > 0 and datetime.now() < end:
            time.sleep(sleep_for)

    print("-" * 50)
    print("Run complete.", sample_number, "samples over", datetime.now() - start)


# Quick check: a short burst of samples at a 10 second interval for about 30 seconds,
# instead of the real 5 minutes over 24 hours. This leaves a small log for the benchmark
# and alarm sections below to use while we build them.
run_monitor(interval=timedelta(seconds=10), duration=timedelta(seconds=30), fresh_start=True)

_wb = load_workbook(LOG_PATH)
print("Log sheets:", _wb.sheetnames)
print("Log system_metrics rows:", _wb["system_metrics"].max_row)
_wb.close()


Run started at 2026-06-11 14:32:49
Interval: 0:00:10  Duration: 0:00:30
--------------------------------------------------
Sample 1 at 14:32:49   CPU 24.8%   Memory 87.4%
Sample 2 at 14:32:59   CPU 28.2%   Memory 88.5%
Sample 3 at 14:33:09   CPU 17.5%   Memory 89.1%
--------------------------------------------------
Run complete. 3 samples over 0:00:30.001831
Log sheets: ['system_metrics', 'disk', 'network', 'processes', 'services']
Log system_metrics rows: 4


## Fixed threshold benchmark and alarm

This is the first and simplest benchmark. Instead of learning what normal looks like, it
uses plain limits set in the config section as the definition of normal: CPU, memory,
disk, GPU and temperature each have a ceiling, the battery has a floor, and a list of
services are expected to always be running. Anything that crosses a limit is a deviation
and becomes an alarm.

The function reads the log, checks every sample against the limits, and returns a list of
alarms. Each alarm is a plain-language row: when it happened, which method raised it, the
resource involved, the reading, the limit it crossed, and a sentence explaining it. These
rows are what later get written to the alarms sheet.

This section also defines `read_log_sheet`, a small helper that loads any sheet of the log
into a list of rows. The other benchmark sections reuse it, so run this section before
them.

The check below runs the fixed threshold benchmark against the short test log and prints
how many alarms it found and the first few.


In [69]:
from openpyxl import load_workbook


def read_log_sheet(sheet_name, path=None):
    """Load one sheet of the log into a list of plain dictionaries keyed by header."""
    path = path or LOG_PATH
    if not path.exists():
        return []
    wb = load_workbook(path, read_only=True, data_only=True)
    if sheet_name not in wb.sheetnames:
        wb.close()
        return []
    rows = list(wb[sheet_name].iter_rows(values_only=True))
    wb.close()
    if not rows:
        return []
    headers = rows[0]
    return [dict(zip(headers, values)) for values in rows[1:]]


def _alarm(time, source, resource, reading, limit, message):
    """Build one plain-language alarm row."""
    return {
        "Time": time,
        "Source": source,
        "Resource": resource,
        "Reading": reading,
        "Limit": limit,
        "Message": message,
    }


def _is_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool)


def check_fixed_thresholds():
    """Flag every sample value that crosses a fixed limit from the config section."""
    alarms = []

    for row in read_log_sheet("system_metrics"):
        time = row.get("Time")

        cpu = row.get("Overall CPU Usage (%)")
        if _is_number(cpu) and cpu > CPU_ALARM_PERCENT:
            alarms.append(_alarm(time, "Fixed threshold", "CPU", cpu, CPU_ALARM_PERCENT,
                "CPU usage {}% is above the {}% limit".format(cpu, CPU_ALARM_PERCENT)))

        memory = row.get("Memory Usage (%)")
        if _is_number(memory) and memory > MEMORY_ALARM_PERCENT:
            alarms.append(_alarm(time, "Fixed threshold", "Memory", memory, MEMORY_ALARM_PERCENT,
                "Memory usage {}% is above the {}% limit".format(memory, MEMORY_ALARM_PERCENT)))

        gpu = row.get("GPU Usage (%)")
        if _is_number(gpu) and gpu > GPU_ALARM_PERCENT:
            alarms.append(_alarm(time, "Fixed threshold", "GPU", gpu, GPU_ALARM_PERCENT,
                "GPU usage {}% is above the {}% limit".format(gpu, GPU_ALARM_PERCENT)))

        temperature = row.get("CPU Temperature (°C)")
        if _is_number(temperature) and temperature > TEMPERATURE_ALARM_C:
            alarms.append(_alarm(time, "Fixed threshold", "Temperature", temperature, TEMPERATURE_ALARM_C,
                "CPU temperature {}C is above the {}C limit".format(temperature, TEMPERATURE_ALARM_C)))

        battery = row.get("Battery Charge (%)")
        plugged = row.get("Power Plugged In")
        if _is_number(battery) and battery < BATTERY_LOW_PERCENT and plugged == "No":
            alarms.append(_alarm(time, "Fixed threshold", "Battery", battery, BATTERY_LOW_PERCENT,
                "Battery charge {}% is below the {}% floor while on battery power".format(
                    battery, BATTERY_LOW_PERCENT)))

    for row in read_log_sheet("disk"):
        usage = row.get("Disk Usage (%)")
        if _is_number(usage) and usage > DISK_ALARM_PERCENT:
            drive = row.get("Drive")
            alarms.append(_alarm(row.get("Time"), "Fixed threshold", "Disk " + str(drive),
                usage, DISK_ALARM_PERCENT,
                "Drive {} usage {}% is above the {}% limit".format(drive, usage, DISK_ALARM_PERCENT)))

    for row in read_log_sheet("services"):
        name = row.get("Service Name")
        status = row.get("Status")
        if name in REQUIRED_SERVICES and status != "running":
            alarms.append(_alarm(row.get("Time"), "Fixed threshold", "Service " + str(name),
                status, "running",
                "Required service {} is {} instead of running".format(name, status)))

    return alarms


# Quick check: run the fixed threshold benchmark against the short test log.
_fixed_alarms = check_fixed_thresholds()
print("Fixed threshold alarms found:", len(_fixed_alarms))
for _a in _fixed_alarms[:5]:
    print("  ", _a["Time"], "-", _a["Message"])


Fixed threshold alarms found: 0


## Statistical baseline benchmark and alarm

This benchmark learns what normal looks like instead of being told. It takes the first
part of the run as normal operating conditions (the fraction is set in the config), and
for each metric it works out the average and the spread (standard deviation) of those
normal samples. The normal range is the average plus or minus a few standard deviations,
again set in the config. Any later sample that falls outside its normal range is a
deviation and becomes an alarm.

This is closer to the real requirement than fixed limits, because the normal range is
specific to this machine and this workload rather than a guessed number. It needs enough
normal samples to be meaningful, which the full 24 hour run provides. A metric with too
few readings to learn from is skipped rather than guessed.

`compute_baseline` works out the normal range for each metric and is shared with the saved
baseline section below. `check_statistical_baseline` applies it and returns the alarms.

The check below runs against the short test log, which has too few samples to learn from,
so it then shows a small demonstration on made-up data: several normal CPU readings
followed by one clear spike, which the benchmark flags.


In [70]:
import statistics

# Metrics the statistical benchmark learns a normal range for. Non-numeric readings
# such as an Unavailable temperature are skipped automatically.
BASELINE_METRICS = [
    "Overall CPU Usage (%)",
    "Memory Usage (%)",
    "GPU Usage (%)",
    "CPU Temperature (°C)",
]

# Need at least this many normal readings of a metric to trust its range.
MIN_BASELINE_SAMPLES = 3


def compute_baseline(rows, metrics=None):
    """Work out the normal range (mean and spread) for each metric from normal rows."""
    metrics = metrics if metrics is not None else BASELINE_METRICS
    baseline = {}
    for metric in metrics:
        values = [row.get(metric) for row in rows if _is_number(row.get(metric))]
        if len(values) < MIN_BASELINE_SAMPLES:
            continue
        mean = statistics.mean(values)
        spread = statistics.pstdev(values)
        baseline[metric] = {
            "mean": round(mean, 2),
            "spread": round(spread, 2),
            "low": round(mean - BASELINE_STD_MULTIPLIER * spread, 2),
            "high": round(mean + BASELINE_STD_MULTIPLIER * spread, 2),
        }
    return baseline


def check_statistical_baseline(rows=None):
    """Learn a normal range from the first part of the data and flag later outliers."""
    rows = rows if rows is not None else read_log_sheet("system_metrics")
    if len(rows) < MIN_BASELINE_SAMPLES + 1:
        return [], {}

    split = max(1, int(len(rows) * BASELINE_FRACTION))
    baseline = compute_baseline(rows[:split])
    if not baseline:
        return [], {}

    alarms = []
    for row in rows[split:]:
        time = row.get("Time")
        for metric, normal in baseline.items():
            value = row.get(metric)
            if not _is_number(value):
                continue
            if value < normal["low"] or value > normal["high"]:
                normal_range = "{} to {}".format(normal["low"], normal["high"])
                alarms.append(_alarm(time, "Statistical baseline", metric, value, normal_range,
                    "{} reading {} is outside the normal range {}".format(
                        metric, value, normal_range)))
    return alarms, baseline


# Quick check against the short test log (likely too few samples to learn from).
_stat_alarms, _stat_baseline = check_statistical_baseline()
print("Real log: learned normal ranges for", list(_stat_baseline.keys()))
print("Real log statistical alarms:", len(_stat_alarms))

# Demonstration on made-up data so the logic is visible: twelve normal CPU readings,
# then one clear spike that should be flagged.
_demo_values = [20, 22, 19, 21, 23, 20, 18, 22, 21, 19, 20, 22, 96]
_demo_rows = [{"Time": "demo-{}".format(i), "Overall CPU Usage (%)": v}
              for i, v in enumerate(_demo_values)]
_demo_alarms, _demo_baseline = check_statistical_baseline(_demo_rows)
_cpu_normal = _demo_baseline.get("Overall CPU Usage (%)", {})
print("\nDemonstration:")
print("  Learned normal CPU range:", _cpu_normal.get("low"), "to", _cpu_normal.get("high"))
for _a in _demo_alarms:
    print("  DEMO alarm:", _a["Message"])


Real log: learned normal ranges for []
Real log statistical alarms: 0

Demonstration:
  Learned normal CPU range: 16.59 to 24.07
  DEMO alarm: Overall CPU Usage (%) reading 96 is outside the normal range 16.59 to 24.07


## Saved baseline benchmark and alarm

The statistical benchmark above learns normal from inside the same run. This one makes the
benchmark reusable. The idea matches how the requirement describes it: run the monitor once
in known-normal conditions to produce a saved benchmark file, then on later runs compare
against that saved profile instead of relearning it every time.

The logic is simple. If no saved benchmark file exists yet, it builds one from the current
data and writes it to the benchmark file set in the config. If a saved benchmark already
exists, it loads that file and checks every sample against it, flagging anything outside
the saved normal range. This way a stable definition of normal can be captured once and
reused across days.

It reuses `compute_baseline` from the statistical section, so run that section first.

The check below uses a temporary benchmark file so the real one is not touched. It shows
the two states: a first run with no file that creates and saves the benchmark, then a
second run that loads the saved file and flags a clear spike against it.


In [71]:
import json


def save_benchmark(benchmark, path=None):
    """Write a learned normal profile to a benchmark file."""
    path = path or BENCHMARK_PATH
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(benchmark, handle, indent=2)


def load_benchmark(path=None):
    """Read a saved normal profile, or None if there is no saved file yet."""
    path = path or BENCHMARK_PATH
    if not path.exists():
        return None
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def check_saved_baseline(rows=None, path=None):
    """Compare samples against a saved benchmark, creating it the first time.

    Returns the alarms, the benchmark used, and whether it was created this call.
    """
    rows = rows if rows is not None else read_log_sheet("system_metrics")
    benchmark = load_benchmark(path)
    created = False

    if benchmark is None:
        # No saved benchmark yet. Treat the given data as normal and save it.
        benchmark = compute_baseline(rows)
        if not benchmark:
            return [], None, False
        save_benchmark(benchmark, path)
        created = True

    alarms = []
    for row in rows:
        time = row.get("Time")
        for metric, normal in benchmark.items():
            value = row.get(metric)
            if not _is_number(value):
                continue
            if value < normal["low"] or value > normal["high"]:
                normal_range = "{} to {}".format(normal["low"], normal["high"])
                alarms.append(_alarm(time, "Saved baseline", metric, value, normal_range,
                    "{} reading {} is outside the saved normal range {}".format(
                        metric, value, normal_range)))
    return alarms, benchmark, created


# Quick check on a temporary benchmark file so the real one is untouched.
_demo_path = Path("outputs") / "_demo_benchmark.json"
if _demo_path.exists():
    _demo_path.unlink()

_normal_rows = [{"Time": "n{}".format(i), "Overall CPU Usage (%)": v}
                for i, v in enumerate([20, 22, 19, 21, 23, 20, 18, 22])]
_a1, _b1, _created1 = check_saved_baseline(_normal_rows, path=_demo_path)
_cpu = _b1["Overall CPU Usage (%)"]
print("First run created benchmark:", _created1,
      "  CPU normal range:", _cpu["low"], "to", _cpu["high"])

_new_rows = [{"Time": "m{}".format(i), "Overall CPU Usage (%)": v}
             for i, v in enumerate([21, 20, 97, 19])]
_a2, _b2, _created2 = check_saved_baseline(_new_rows, path=_demo_path)
print("Second run created benchmark:", _created2, "  alarms:", len(_a2))
for _a in _a2:
    print("  DEMO alarm:", _a["Message"])

_demo_path.unlink()
print("Demo benchmark file cleaned up.")


First run created benchmark: True   CPU normal range: 15.9 to 25.35
Second run created benchmark: False   alarms: 1
  DEMO alarm: Overall CPU Usage (%) reading 97 is outside the saved normal range 15.9 to 25.35
Demo benchmark file cleaned up.


## Concluding combined alarm

This is the headline of the whole project. It runs all three benchmark methods together,
the fixed thresholds, the statistical baseline and the saved baseline, gathers every alarm
they raise, and combines them into one result. It then writes all of those alarms to a
dedicated alarms sheet in both the dashboard and the log, each alarm timestamped, alongside
the resource it concerns, the reading, the limit or normal range it crossed, and a plain
sentence describing it.

Finally it prints the single overall verdict: all clear if nothing deviated, or an alarm
with the count and a short breakdown of where the deviations came from. This is the one
concluding alarm that answers the actual requirement: has anything strayed from normal.

The check below runs the full concluding alarm against the short test log and writes the
alarms sheet into both Excel files. It removes any benchmark file it creates from this tiny
test data afterwards, so the real benchmark is built later from a real run.


In [72]:
from collections import Counter

from openpyxl import load_workbook

ALARM_HEADERS = ["Time", "Source", "Resource", "Reading", "Limit", "Message"]


def gather_all_alarms():
    """Run all three benchmark methods and return every alarm they raise, combined."""
    fixed = check_fixed_thresholds()
    statistical, _ = check_statistical_baseline()
    saved, _, _ = check_saved_baseline()
    return fixed + statistical + saved


def write_alarms_sheet(alarms):
    """Write the combined alarms to an alarms sheet in both Excel files."""
    for path in (DASHBOARD_PATH, LOG_PATH):
        if not path.exists():
            continue
        wb = load_workbook(path)
        if "alarms" in wb.sheetnames:
            del wb["alarms"]
        ws = wb.create_sheet("alarms")
        ws.append(ALARM_HEADERS)
        for alarm in alarms:
            ws.append([alarm.get(header) for header in ALARM_HEADERS])
        wb.save(path)


def conclude_alarms():
    """Combine every method's alarms, write them to Excel, and print the verdict."""
    alarms = gather_all_alarms()
    write_alarms_sheet(alarms)

    print("=" * 60)
    print("CONCLUDING ALARM")
    print("=" * 60)
    if not alarms:
        print("ALL CLEAR. No deviations from normal were detected.")
    else:
        print("ALARM. {} deviations from normal were detected.".format(len(alarms)))
        print("")
        print("By detection method:")
        for source, count in Counter(a["Source"] for a in alarms).most_common():
            print("   {:<22} {}".format(source, count))
        print("")
        print("Most affected resources:")
        for resource, count in Counter(a["Resource"] for a in alarms).most_common(5):
            print("   {:<22} {}".format(resource, count))
    print("")
    print("All alarms written to the alarms sheet in the dashboard and the log.")
    return alarms


# Quick check: run the concluding alarm against the short test log.
_remember_benchmark = BENCHMARK_PATH.exists()
_all_alarms = conclude_alarms()

# Clean up a benchmark created from this tiny test data so the real run builds its own.
if not _remember_benchmark and BENCHMARK_PATH.exists():
    BENCHMARK_PATH.unlink()
    print("Test benchmark file cleaned up.")


CONCLUDING ALARM
ALL CLEAR. No deviations from normal were detected.

All alarms written to the alarms sheet in the dashboard and the log.
Test benchmark file cleaned up.


## Charts at end of run

This section turns the logged history into trend graphs, built once at the end of the run.
It reads the log and draws how the key numbers moved over time: CPU and memory usage,
battery charge, GPU usage, and network data sent and received per sample.

It produces the graphs in two forms so they are useful to anyone. Native Excel line charts
are added to the dashboard on a charts sheet, backed by a trends data sheet, so they open
and update right inside Excel. The same graphs are also saved as PNG image files in the
charts folder, ready to drop straight into a report or email.

`build_charts` does both. It is called automatically at the end of the run in the final
run cell, but it can also be run on its own at any time against an existing log.

The check below builds the charts from the short test log so you can confirm the image
files appear and the dashboard gains its charts sheet.


In [73]:
import matplotlib
matplotlib.use("Agg")  # draw to image files, no popup window needed
import matplotlib.pyplot as plt

from openpyxl import load_workbook
from openpyxl.chart import LineChart, Reference

# Columns of the trend table, in order. Column 1 is Time, the rest are the metrics.
TREND_COLUMNS = [
    "Time",
    "CPU Usage (%)",
    "Memory Usage (%)",
    "Battery Charge (%)",
    "GPU Usage (%)",
    "Data Sent (MB)",
    "Data Received (MB)",
]


def build_trend_table():
    """Read the log and build one trend row per sample with the headline metrics."""
    system = read_log_sheet("system_metrics")
    network = read_log_sheet("network")

    # Network is per interface, so add up the per-sample amounts for each timestamp.
    sent_by_time = {}
    recv_by_time = {}
    for row in network:
        time = row.get("Time")
        sent = row.get("Data Sent Since Last Sample (MB)")
        recv = row.get("Data Received Since Last Sample (MB)")
        if _is_number(sent):
            sent_by_time[time] = sent_by_time.get(time, 0) + sent
        if _is_number(recv):
            recv_by_time[time] = recv_by_time.get(time, 0) + recv

    def numeric_or_none(value):
        return value if _is_number(value) else None

    table = []
    for row in system:
        time = row.get("Time")
        table.append({
            "Time": time,
            "CPU Usage (%)": numeric_or_none(row.get("Overall CPU Usage (%)")),
            "Memory Usage (%)": numeric_or_none(row.get("Memory Usage (%)")),
            "Battery Charge (%)": numeric_or_none(row.get("Battery Charge (%)")),
            "GPU Usage (%)": numeric_or_none(row.get("GPU Usage (%)")),
            "Data Sent (MB)": round(sent_by_time.get(time, 0), 3),
            "Data Received (MB)": round(recv_by_time.get(time, 0), 3),
        })
    return table


def _save_png_charts(table):
    """Save the trend graphs as PNG image files in the charts folder."""
    if not table:
        return []
    times = [row["Time"] for row in table]
    saved = []

    def plot(filename, title, ylabel, series):
        figure, axis = plt.subplots(figsize=(10, 4))
        plotted = False
        for label, key in series:
            points = [(t, row[key]) for t, row in zip(times, table) if _is_number(row.get(key))]
            if points:
                xs, ys = zip(*points)
                axis.plot(xs, ys, marker="o", label=label)
                plotted = True
        axis.set_title(title)
        axis.set_xlabel("Time")
        axis.set_ylabel(ylabel)
        if plotted:
            axis.legend()
        figure.autofmt_xdate()
        figure.tight_layout()
        out_path = CHARTS_DIR / filename
        figure.savefig(out_path, dpi=100)
        plt.close(figure)
        saved.append(out_path)

    plot("cpu_memory.png", "CPU and Memory Usage Over Time", "Percent (%)",
         [("CPU", "CPU Usage (%)"), ("Memory", "Memory Usage (%)")])
    plot("battery.png", "Battery Charge Over Time", "Percent (%)",
         [("Battery", "Battery Charge (%)")])
    plot("gpu.png", "GPU Usage Over Time", "Percent (%)",
         [("GPU", "GPU Usage (%)")])
    plot("network.png", "Network Data Per Sample Over Time", "Megabytes (MB)",
         [("Sent", "Data Sent (MB)"), ("Received", "Data Received (MB)")])
    return saved


def _add_excel_charts(table):
    """Add native Excel line charts to the dashboard, backed by a trends data sheet."""
    if not table or not DASHBOARD_PATH.exists():
        return
    wb = load_workbook(DASHBOARD_PATH)
    for name in ("trends", "charts"):
        if name in wb.sheetnames:
            del wb[name]

    trends = wb.create_sheet("trends")
    trends.append(TREND_COLUMNS)
    for row in table:
        trends.append([row.get(column) for column in TREND_COLUMNS])

    charts = wb.create_sheet("charts")
    last_row = len(table) + 1
    categories = Reference(trends, min_col=1, min_row=2, max_row=last_row)

    def line(title, min_col, max_col, anchor):
        chart = LineChart()
        chart.title = title
        chart.height = 8
        chart.width = 16
        data = Reference(trends, min_col=min_col, max_col=max_col, min_row=1, max_row=last_row)
        chart.add_data(data, titles_from_data=True)
        chart.set_categories(categories)
        charts.add_chart(chart, anchor)

    # Trend columns: B=CPU, C=Memory, D=Battery, E=GPU, F=Sent, G=Received.
    line("CPU and Memory Usage (%)", 2, 3, "A1")
    line("Battery Charge (%)", 4, 4, "A18")
    line("GPU Usage (%)", 5, 5, "A35")
    line("Network Data Per Sample (MB)", 6, 7, "A52")
    wb.save(DASHBOARD_PATH)


def build_charts():
    """Build the trend graphs as PNG files and as Excel charts in the dashboard."""
    table = build_trend_table()
    saved = _save_png_charts(table)
    _add_excel_charts(table)
    print("Charts built from", len(table), "samples.")
    print("PNG images saved:", [path.name for path in saved])
    if table:
        print("Trends and charts sheets added to the dashboard.")
    return table


# Quick check: build the charts from the short test log.
build_charts()


Charts built from 3 samples.
PNG images saved: ['cpu_memory.png', 'battery.png', 'gpu.png', 'network.png']
Trends and charts sheets added to the dashboard.


[{'Time': datetime.datetime(2026, 6, 11, 14, 32, 49, 893000),
  'CPU Usage (%)': 24.8,
  'Memory Usage (%)': 87.4,
  'Battery Charge (%)': 81,
  'GPU Usage (%)': 3.8,
  'Data Sent (MB)': 0,
  'Data Received (MB)': 0},
 {'Time': datetime.datetime(2026, 6, 11, 14, 32, 59, 893000),
  'CPU Usage (%)': 28.2,
  'Memory Usage (%)': 88.5,
  'Battery Charge (%)': 80,
  'GPU Usage (%)': 2.1,
  'Data Sent (MB)': 0.01,
  'Data Received (MB)': 0.004},
 {'Time': datetime.datetime(2026, 6, 11, 14, 33, 9, 892000),
  'CPU Usage (%)': 17.5,
  'Memory Usage (%)': 89.1,
  'Battery Charge (%)': 80,
  'GPU Usage (%)': 3.2,
  'Data Sent (MB)': 0.019,
  'Data Received (MB)': 0.006}]

## Run the full monitor

This is the cell you run to do the job. `run_full_monitor` ties the whole notebook
together in order: it collects samples for the configured time, then runs the concluding
alarm that benchmarks against normal and writes the alarms sheet, then builds the trend
charts. When it finishes, the outputs folder holds the dashboard, the log, the alarms, the
benchmark file and the chart images.

With no arguments it uses the real settings from the config section, every 5 minutes for 24
hours. That is the real run. To start it, run this in a cell:

    run_full_monitor()

You can also pass a short interval and duration for a quick demonstration, which is what
the check below does. The demo proves the full pipeline end to end in under a minute and
leaves real outputs you can open, while removing the tiny demo benchmark so the first real
run builds its own.


In [74]:
def run_full_monitor(interval=None, duration=None, fresh_start=True):
    """Collect samples, raise the concluding alarm, then build the charts, in order."""
    run_monitor(interval=interval, duration=duration, fresh_start=fresh_start)
    print("")
    conclude_alarms()
    print("")
    build_charts()
    print("")
    print("Done. The dashboard, log, alarms, benchmark and charts are in the outputs folder.")


# Quick check: a full end to end demonstration at a 10 second interval for about 30
# seconds, instead of the real 5 minutes over 24 hours.
_remember_benchmark = BENCHMARK_PATH.exists()
run_full_monitor(interval=timedelta(seconds=10), duration=timedelta(seconds=30))

# Remove the benchmark built from this tiny demo so the first real run builds its own.
if not _remember_benchmark and BENCHMARK_PATH.exists():
    BENCHMARK_PATH.unlink()
    print("Demo benchmark file cleaned up.")


Run started at 2026-06-11 14:33:22
Interval: 0:00:10  Duration: 0:00:30
--------------------------------------------------
Sample 1 at 14:33:22   CPU 25.8%   Memory 89.1%
Sample 2 at 14:33:32   CPU 30.3%   Memory 89.2%
Sample 3 at 14:33:42   CPU 51.7%   Memory 85.1%
--------------------------------------------------
Run complete. 3 samples over 0:00:30.000469

CONCLUDING ALARM
ALL CLEAR. No deviations from normal were detected.

All alarms written to the alarms sheet in the dashboard and the log.

Charts built from 3 samples.
PNG images saved: ['cpu_memory.png', 'battery.png', 'gpu.png', 'network.png']
Trends and charts sheets added to the dashboard.

Done. The dashboard, log, alarms, benchmark and charts are in the outputs folder.
Demo benchmark file cleaned up.
